# TP2 – Comptage de mots avec PySpark

**Objectif :** reprendre le comptage d'occurrences de mots du TP1 (Python simple, pseudo-MapReduce, Hadoop MapReduce) avec **Apache Spark** via son API Python, **PySpark**.

On implémente le WordCount de trois façons :

1. avec l'API **RDD** (`flatMap` → `map` → `reduceByKey`), équivalent direct de MapReduce ;
2. avec l'API **DataFrame** (`split` → `explode` → `groupBy().count()`) ;
3. avec **Spark SQL**.

On compare ensuite les résultats et les temps d'exécution avec ceux du TP1.

**Environnement :** Python 3.11, PySpark 3.5, Java 17, Spark en mode local (`local[*]`).

## 1. Principe d'Apache Spark

Spark est un moteur de calcul distribué qui reprend le modèle MapReduce en corrigeant ses principales limites :

- **Calcul en mémoire** : les résultats intermédiaires restent en RAM au lieu d'être écrits sur disque (HDFS) entre chaque étape, comme c'est le cas avec Hadoop MapReduce.
- **Évaluation paresseuse** : les *transformations* (`map`, `flatMap`, `filter`, `reduceByKey`…) ne font que construire un plan d'exécution (DAG). Le calcul n'est lancé qu'au moment d'une *action* (`collect`, `count`, `show`…).
- **API de haut niveau** : RDD (bas niveau, proche de MapReduce), DataFrame et SQL (optimisés automatiquement par l'optimiseur *Catalyst*).

| Étape MapReduce (TP1) | Équivalent Spark RDD |
|---|---|
| `mapper` : ligne → `(mot, 1)` | `flatMap(split)` puis `map(lambda m: (m, 1))` |
| `shuffle` : regroupement par clé | effectué automatiquement par `reduceByKey` |
| `reducer` : somme des valeurs | `reduceByKey(lambda a, b: a + b)` |

Spark peut tourner en local (un seul processus, plusieurs threads) ou sur un cluster (YARN, Kubernetes, Standalone), y compris en lisant les données depuis HDFS.

## 2. Installation

Spark nécessite **Java 8, 11 ou 17**. PySpark s'installe ensuite avec `pip` (il embarque Spark, pas besoin d'installation séparée) :

```bash
# Java (Manjaro)
sudo pacman -S jdk17-openjdk

# PySpark
pip install pyspark==3.5.3 pandas matplotlib
```

## 3. Préparation des données

Même fichier `mots.txt` que dans le TP1 (85 mots, même distribution), généré s'il n'existe pas.

In [ ]:
import os
import random

FICHIER = "mots.txt"

DISTRIBUTION = {
    "chat": 24, "chien": 11, "arbre": 7, "fleur": 7, "maison": 5,
    "soleil": 5, "montagne": 5, "livre": 4, "ciel": 4, "mer": 3,
    "lune": 3, "route": 3, "oiseau": 2, "jardin": 1, "voiture": 1,
}

if not os.path.exists(FICHIER):
    mots = [m for m, n in DISTRIBUTION.items() for _ in range(n)]
    random.seed(42)
    random.shuffle(mots)
    with open(FICHIER, "w", encoding="utf-8") as f:
        for i in range(0, len(mots), 8):          # 8 mots par ligne
            f.write(" ".join(mots[i:i + 8]) + "\n")
    print(f"{FICHIER} généré ({len(mots)} mots).")
else:
    print(f"{FICHIER} déjà présent, il sera utilisé tel quel.")

## 4. Création de la session Spark

La `SparkSession` est le point d'entrée de PySpark. `local[*]` lance Spark en local en utilisant tous les cœurs disponibles. On mesure le temps de démarrage, qui correspond au lancement de la JVM.

In [ ]:
import sys
import time

# Les workers Python de Spark utilisent le même interpréteur que le notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("WordCount TP2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
duree_demarrage = time.perf_counter() - t0

print("Version de Spark :", spark.version)
print("Master           :", sc.master)
print(f"Démarrage        : {duree_demarrage:.2f} s")

## 5. Version 1 : API RDD

Un **RDD** (*Resilient Distributed Dataset*) est une collection distribuée et immuable. Le pipeline reproduit exactement les étapes *map / shuffle / reduce* du TP1.

In [ ]:
def wordcount_rdd(fichier):
    return (
        sc.textFile(fichier)                       # RDD de lignes
        .flatMap(lambda ligne: ligne.split())      # Map : ligne -> mots
        .map(lambda mot: (mot, 1))                 # Map : mot -> (mot, 1)
        .reduceByKey(lambda a, b: a + b)           # Shuffle + Reduce
    )


rdd_compte = wordcount_rdd(FICHIER)   # rien n'est encore calculé (évaluation paresseuse)

resultats_rdd = dict(rdd_compte.collect())   # action : déclenche le calcul
for mot, nb in sorted(resultats_rdd.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

Le plan d'exécution (lignée du RDD) montre la séparation en deux *stages* autour du shuffle introduit par `reduceByKey` :

In [ ]:
print(rdd_compte.toDebugString().decode())

## 6. Version 2 : API DataFrame

Un **DataFrame** est une table distribuée avec un schéma (colonnes nommées et typées). Les opérations sont optimisées par Catalyst et exécutées dans la JVM, sans passer par des fonctions Python.

In [ ]:
from pyspark.sql import functions as F


def wordcount_df(fichier):
    return (
        spark.read.text(fichier)                                    # colonne "value" = une ligne
        .select(F.explode(F.split(F.trim("value"), r"\s+")).alias("mot"))
        .filter(F.col("mot") != "")
        .groupBy("mot")
        .count()
        .orderBy(F.desc("count"), "mot")
    )


df_compte = wordcount_df(FICHIER)
df_compte.show(truncate=False)

resultats_df = {r["mot"]: r["count"] for r in df_compte.collect()}

Plan physique généré par l'optimiseur Catalyst :

In [ ]:
df_compte.explain()

## 7. Version 3 : Spark SQL

Le même traitement exprimé en SQL, sur une vue temporaire.

In [ ]:
spark.read.text(FICHIER).createOrReplaceTempView("lignes")

REQUETE = """
    SELECT mot, COUNT(*) AS count
    FROM (SELECT explode(split(trim(value), '\\\\s+')) AS mot FROM lignes)
    WHERE mot <> ''
    GROUP BY mot
    ORDER BY count DESC, mot
"""


def wordcount_sql():
    return spark.sql(REQUETE)


sql_compte = wordcount_sql()
sql_compte.show(truncate=False)

resultats_sql = {r["mot"]: r["count"] for r in sql_compte.collect()}

## 8. Vérification

On compare les trois versions Spark avec la version Python simple du TP1.

In [ ]:
def compter_occurrences(fichier):
    """Version 1 du TP1 : Python simple."""
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()
    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur


reference = compter_occurrences(FICHIER)

assert resultats_rdd == reference
assert resultats_df == reference
assert resultats_sql == reference
print("OK : RDD, DataFrame et SQL donnent le même résultat que Python simple,",
      sum(reference.values()), "mots comptés.")

## 9. Comparaison des performances

Chaque méthode est exécutée plusieurs fois (on force le calcul avec `collect()`). La première exécution Spark est plus lente (compilation du plan, démarrage des workers Python), on la mesure donc à part.

In [ ]:
import statistics


def mesurer(fonction, repetitions):
    durees = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        fonction()
        durees.append(time.perf_counter() - t0)
    return statistics.mean(durees), min(durees)


moy_py, min_py = mesurer(lambda: compter_occurrences(FICHIER), repetitions=200)
moy_rdd, min_rdd = mesurer(lambda: wordcount_rdd(FICHIER).collect(), repetitions=10)
moy_df, min_df = mesurer(lambda: wordcount_df(FICHIER).collect(), repetitions=10)
moy_sql, min_sql = mesurer(lambda: wordcount_sql().collect(), repetitions=10)

DUREE_HADOOP_TP1 = 12.0  # secondes, mesurée dans le TP1

print(f"Démarrage Spark  : {duree_demarrage:.2f} s (une seule fois)")
print(f"Python simple    : moyenne {moy_py*1000:8.3f} ms | min {min_py*1000:8.3f} ms")
print(f"Spark RDD        : moyenne {moy_rdd*1000:8.3f} ms | min {min_rdd*1000:8.3f} ms")
print(f"Spark DataFrame  : moyenne {moy_df*1000:8.3f} ms | min {min_df*1000:8.3f} ms")
print(f"Spark SQL        : moyenne {moy_sql*1000:8.3f} ms | min {min_sql*1000:8.3f} ms")
print(f"Hadoop MapReduce : {DUREE_HADOOP_TP1:.1f} s (TP1)")

In [ ]:
import pandas as pd

tableau = pd.DataFrame({
    "Méthode": ["Python simple (TP1)", "Spark RDD", "Spark DataFrame", "Spark SQL", "Hadoop MapReduce (TP1)"],
    "Temps (s)": [moy_py, moy_rdd, moy_df, moy_sql, DUREE_HADOOP_TP1],
    "Remarques": [
        "Lecture directe en mémoire, imbattable sur un petit fichier.",
        "Équivalent MapReduce ; lambdas exécutées par des workers Python.",
        "Optimisé par Catalyst, exécuté dans la JVM.",
        "Même plan d'exécution que le DataFrame.",
        "Lancement d'un job YARN + lectures/écritures HDFS.",
    ],
})
tableau["Temps (s)"] = tableau["Temps (s)"].map(lambda x: f"{x:.4f}" if x < 1 else f"{x:.1f}")
tableau

In [ ]:
import matplotlib.pyplot as plt

noms = ["Python simple", "Spark RDD", "Spark DataFrame", "Spark SQL", "Hadoop MapReduce"]
temps = [moy_py, moy_rdd, moy_df, moy_sql, DUREE_HADOOP_TP1]

fig, ax = plt.subplots(figsize=(8, 4))
barres = ax.bar(noms, temps, color=["#4c78a8", "#f58518", "#e45756", "#72b7b2", "#54a24b"])
ax.set_yscale("log")
ax.set_ylabel("Temps d'exécution (s, échelle log)")
ax.set_title("WordCount : Python, Spark et Hadoop")
for b, t in zip(barres, temps):
    ax.text(b.get_x() + b.get_width() / 2, t, f"{t:.3g} s", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 9.1 Analyse

- Sur un fichier de 85 mots, **Python simple reste de loin le plus rapide** (une fraction de milliseconde) : Spark doit planifier un job, découper les données en partitions et organiser un shuffle, ce qui a un coût fixe quelle que soit la taille des données.
- **Les API DataFrame et SQL** (de l'ordre de quelques centaines de millisecondes) restent entièrement dans la JVM et bénéficient de l'optimiseur Catalyst. Elles produisent **le même plan d'exécution** : le choix entre les deux est une question de lisibilité.
- **L'API RDD est nettement plus lente** (de l'ordre de la seconde) : les lambdas Python sont exécutées par des processus Python séparés, avec sérialisation des données entre la JVM et Python. Sous Windows ce coût est accentué, car les workers Python sont relancés à chaque job (pas de `fork`). C'est pourquoi on privilégie les DataFrames en PySpark.
- **Spark reste plus rapide que Hadoop MapReduce** (~12 s) : une fois la session démarrée, chaque job réutilise la même JVM et garde les données intermédiaires en mémoire, alors que Hadoop lance un nouveau job YARN et écrit sur HDFS entre map et reduce.
- Le **démarrage de la session** (JVM, quelques secondes) n'est payé qu'une seule fois, contrairement à Hadoop où chaque job a sa propre latence de lancement.

## 10. Conclusion

PySpark permet d'écrire le WordCount en quelques lignes, avec une API RDD qui reprend directement le modèle MapReduce du TP1 et des API DataFrame/SQL de plus haut niveau, optimisées automatiquement. Grâce au calcul en mémoire et à la réutilisation de la session, Spark supprime l'essentiel de la latence de Hadoop MapReduce tout en conservant sa capacité à passer à l'échelle sur un cluster. Pour un très petit fichier, un script Python séquentiel reste toutefois le plus efficace : les outils distribués ne deviennent intéressants que lorsque le volume de données dépasse la capacité d'une seule machine.

In [ ]:
spark.stop()